# Text Rendering: OCR-Exploit [LB=0.305]

## Overview  

* The pre-training dataset used in this competition's evaluation model [1] includes OCR text-image pairs [2]. **Therefore, drawing descriptive text directly on SVG images can produce meaningful similarity with the input texts.**  
* This notebook demonstrates how to leverage this observation to improve CLIP score.

## Method  

### 1. OCR-Exploit

* In this competition, the allowed SVG format restricts direct text rendering. To overcome this, the model must represent text using permitted SVG tags.
* This notebook demonstrates a method to convert font glyphs into SVG paths, enabling text visualization while adhering to the constraints.  

### 2. Input Space Exploration  

* Due to the efficiency of rule-based rendering, it is possible to explore a larger input space (i.e., various SVG configurations) during the test phase.  
* This notebook showcases an approach to optimize these parameters:
    * search **maximum valid text length** within allowed SVG size (<= 10,000)
    * search optimal **background colors** (24 colors)

## Result

**CLIP Score**

* **Local Validation Score (train.csv) = 0.285**  
* **LB = 0.305**  

**Efficiency**

* **Submission Time**: **2,527 sec** (including setup) -> **~5.0 sec/instance**

## Reference  

- [1] google/siglip-so400m-patch14-384, https://huggingface.co/google/siglip-so400m-patch14-384  
- [2] PaLI: A Jointly-Scaled Multilingual Language-Image Model, Sep 2022, https://arxiv.org/abs/2209.06794

In [ ]:
#| default_exp core

In [ ]:
#| export
import gc
import io
import re
from dataclasses import dataclass
from pathlib import Path
from statistics import mean

import cairosvg
import kagglehub
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModel

svg_constraints = kagglehub.package_import('metric/svg-constraints')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class ParticipantVisibleError(Exception):
    pass


def score(
    solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str
) -> float:
    """Calculates a fidelity score by comparing generated SVG images to target text descriptions using a CLIP model.

        This function takes two pandas DataFrames, one containing target descriptions and another containing generated SVGs.
        It uses an `SVGEvaluator` to convert SVGs to PNG images and then calculates a CLIP-based similarity score between the image
        and the corresponding description. The final score is the mean of these similarities.

        Parameters
        ----------
        solution : pd.DataFrame
            A DataFrame containing target text descriptions. Must have a column named 'description'.
        submission : pd.DataFrame
            A DataFrame containing generated SVG strings. Must have a column named 'svg'.
        row_id_column_name : str
            The name of the column containing row identifiers. This column is removed before scoring.

        Returns
        -------
        float
            The mean fidelity score (a value between 0 and 1) representing the average similarity between the generated SVGs and their descriptions.
            A higher score indicates better fidelity.

        Raises
        ------
        ParticipantVisibleError
            If the 'svg' column in the submission DataFrame is not of string type or if validation of the SVG fails.

        Examples
        --------
        >>> import pandas as pd
        >>> solution = pd.DataFrame({
        ...     'id': [0, 1],
        ...     'description': ['red ball', 'swimming pool']
        ... })
        >>> submission = pd.DataFrame({
        ...     'id': [0, 1],
        ...     'svg': ['<svg viewBox="0 0 100 100"><circle cx="50" cy="50" r="40" fill="red"/></svg>',
        ...         '<svg viewBox="0 0 100 100"><rect x="10" y="10" width="80" height="80" fill="blue"/></svg>']
        ... })
        >>> score(solution, submission, 'id')
        0...
    """
    # Validate
    del solution[row_id_column_name], submission[row_id_column_name]
    if not pd.api.types.is_string_dtype(submission.loc[:, 'svg']):
        raise ParticipantVisibleError('svg must be a string.')
    # Check that SVG code meets defined constraints
    constraints = svg_constraints.SVGConstraints()
    try:
        for svg in submission.loc[:, 'svg']:
            constraints.validate_svg(svg)
    except:
        raise ParticipantVisibleError('SVG code violates constraints.')

    # Score
    evaluator = SVGEvaluator()
    results = []
    for svg, description in zip(
        submission.loc[:, 'svg'], solution.loc[:, 'description'], strict=True
    ):
        try:
            similarity = evaluator.evaluate_svg(
                svg,
                "SVG illustration of " + description,
            )
            results.append(similarity)
        except:
            raise ParticipantVisibleError('SVG failed to score.')

    fidelity = mean(results)
    return float(fidelity)


class SVGEvaluator:
    """Evaluates SVG images based on their similarity to a given text description using CLIP.

    This class handles SVG validation, conversion to PNG, and CLIP-based scoring.

    Attributes
    ----------
    device : str
        The device used for CLIP (either 'cuda' if available or 'cpu').
    model : model
    preprocess : callable
        The preprocessing function for images used by the CLIP model.

    Parameters
    ----------
    model_name : str, default='google-siglip-so400m-patch14-384'
        The name of the CLIP model to load.
    constraints : SVGConstraints, optional
        The constraints to use for SVG validation. If None, default constraints are used.
    """

    def __init__(self, model_path):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model_path = Path(model_path)
        self.model = AutoModel.from_pretrained(self.model_path).to(self.device)
        self.processor = AutoProcessor.from_pretrained(self.model_path)

    def svg_to_png(self, svg_code: str, size: tuple = (384, 384)) -> Image.Image:
        """
        Converts an SVG string to a PNG image using CairoSVG.

        If the SVG does not define a `viewBox`, it will add one using the provided size.

        Parameters
        ----------
        svg_code : str
            The SVG string to convert.
        size : tuple[int, int], default=(384, 384)
            The desired size of the output PNG image (width, height).

        Returns
        -------
        PIL.Image.Image
            The generated PNG image.
        """
        # Ensure SVG has proper size attributes
        if 'viewBox' not in svg_code:
            svg_code = svg_code.replace(
                '<svg', f'<svg viewBox="0 0 {size[0]} {size[1]}"'
            )

        # Convert SVG to PNG
        png_data = cairosvg.svg2png(bytestring=svg_code.encode('utf-8'))
        return Image.open(io.BytesIO(png_data)).convert('RGB').resize(size)

    def evaluate_svg(self, svg_code: str, target_class: str) -> float:
        """
        Evaluates the fidelity of an SVG image against a target description using SigLIP.

        This method validates the SVG, converts it to a PNG, preprocesses it, and calculates the SigLIP-based similarity score with the provided description.

        Parameters
        ----------
        svg_code : str
            The SVG string to evaluate.
        target_class : str
            The text description that the SVG should represent.

        Returns
        -------
        float
            The mean similarity score (a value between 0 and 1) representing the match between the SVG and its description.

        """
        # Convert SVG to PNG
        image = self.svg_to_png(svg_code)
        # Preprocess image and text
        inputs = self.processor(
            text=[target_class], images=image, padding="max_length", return_tensors="pt"
        ).to(self.device)

        # Get features and normalize
        with torch.no_grad():
            outputs = self.model(**inputs)
            image_features = outputs.image_embeds
            text_features = outputs.text_embeds

            # Normalize features
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features /= text_features.norm(dim=-1, keepdim=True)

            # Calculate similarity scores
            similarities = (image_features @ text_features.T).squeeze()

        return similarities.item()

    def clear_gpu_memory(self) -> None:
        """Clears GPU memory by deleting references and emptying caches."""
        if not torch.cuda.is_available():
            return

        # Delete model if it exists
        if hasattr(self, 'model'):
            del self.model

        # Run garbage collection
        gc.collect()

        # Clear CUDA cache and reset memory stats
        with DEVICE:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            torch.cuda.reset_peak_memory_stats()

In [ ]:
#| export
import logging

logging.basicConfig(level=logging.INFO, force=True)

## Visualize Sample Text

In [ ]:
#| export
import freetype
import textwrap


def _render_char(face, char, x, y, font_size, px):
    # Return None for spaces; compute glyph path and advance width otherwise.
    if char == " ":
        return None, font_size // 2

    face.load_char(char, freetype.FT_LOAD_NO_BITMAP | freetype.FT_LOAD_TARGET_NORMAL)
    glyph = face.glyph
    pts = [
        (int(round(p[0] / px + x)), int(round(y - p[1] / px)))
        for p in glyph.outline.points
    ]
    cmds = []
    for i, tag in enumerate(glyph.outline.tags):
        if tag & freetype.FT_CURVE_TAG_ON:
            cmds.append(f"L{pts[i][0]},{pts[i][1]}")
        elif tag & freetype.FT_CURVE_TAG_CONIC:
            nxt = pts[(i + 1) % len(pts)]
            cmds.append(f"Q{pts[i][0]},{pts[i][1]},{nxt[0]},{nxt[1]}")
    if cmds:
        cmds[0] = "M" + cmds[0][1:]  # Convert first command to Move (M)
        cmds.append("Z")
        return "".join(cmds), glyph.advance.x / px
    return None, glyph.advance.x / px


def generate_minimal_svg(text, face, font_size=28, wrap_width=20, svg_size=384,
                         line_height_ratio=1.2, bgcolor="gray", text_color="black", px=64):
    face.set_char_size(font_size * px)
    line_height = font_size * line_height_ratio
    lines = textwrap.fill(text, wrap_width).splitlines()
    text_height = len(lines) * line_height

    elems = [f'<rect width="{svg_size}" height="{svg_size}" fill="{bgcolor}"/>']
    paths = []  # Aggregate path commands for all characters.
    x0 = font_size // 8
    y0 = (svg_size - text_height) // 2 + int(line_height // 2)
    y = y0

    for line in lines:
        x = x0
        for ch in line:
            d, adv = _render_char(face, ch, x, y, font_size, px)
            if d:
                paths.append(d)
            x += adv
        y += line_height

    if paths:
        elems.append(f'<path d="{"".join(paths)}" fill="{text_color}"/>')
    return f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {svg_size} {svg_size}">{"".join(elems)}</svg>'

def svg_to_html(svg_code, width="384px"):
    return f'<div style="width:{width};">{svg_code}</div>'

if __name__ == "__main__":
    import logging
    logging.basicConfig(level=logging.INFO)
    from IPython.display import HTML
    import kagglehub

    font_path = kagglehub.dataset_download('tatamikenn/dejavu-fonts-ttf-2-37', path="dejavu-fonts-ttf-2.37/ttf/DejaVuSans-Bold.ttf")
    face = freetype.Face(font_path)
    sample_text = "This is a sample text to generate SVG output."
    svg_output = generate_minimal_svg(sample_text, face)
    logging.info(svg_output)
    logging.info("svg size: {}".format(len(svg_output.encode("utf-8"))))
    display(HTML(svg_to_html(svg_output)))

The code above converts sample text into an SVG image.  
Although some parts of the characters are corrupted, the output text remains human-readable.

In [ ]:
#| export
COLOR_NAMES = [
    "red",
    "green",
    "blue",
    "yellow",
    "purple",
    "orange",
    "pink",
    "brown",
    "gray",
    "cyan",
    "magenta",
    "lime",
    "indigo",
    "violet",
    "gold",
    "silver",
    "beige",
    "maroon",
    "navy",
    "olive",
    "teal",
    "coral",
    "black",
    "white",
]

def extract_color(text, default_color="gray"):
    text_lower = text.lower()
    for color in COLOR_NAMES:
        if color in text_lower:
            return color

    return default_color


def predict(text, evaluator, validator, gen_func, min_clip_len=65, max_clip_len=90, inc=5):
    assert min_clip_len < max_clip_len

    input_len = len(text)
    default_bgcolor = extract_color(text)

    def find_max_valid_clip():
        if input_len <= min_clip_len:
            svg_code = gen_func(text, bgcolor=default_bgcolor)
            return svg_code, input_len

        low = min_clip_len
        high = min(input_len, max_clip_len)
        last_success_svg_code = None

        while low <= high:
            mid = (low + high) // 2
            svg_candidate = gen_func(text[:mid], bgcolor=default_bgcolor)
            try:
                validator.validate_svg(svg_candidate)
                last_success_svg_code = svg_candidate
                low = mid + 1
            except Exception as e:
                high = mid - 1
        return last_success_svg_code, high

    svg_code, clip_len = find_max_valid_clip()
        
    best_score = float("-inf")
    best_svg_code = svg_code
    best_color = default_bgcolor

    for bgcolor in COLOR_NAMES:
        svg_candidate = gen_func(text[:clip_len], bgcolor=bgcolor)
        score = evaluator.evaluate_svg(svg_candidate, text)
        if score > best_score:
            best_score = score
            best_svg_code = svg_candidate
            best_color = bgcolor
    return best_svg_code, best_score, clip_len, best_color

In [ ]:
#| export
import concurrent
import io
import logging
import re
from functools import partial
from dataclasses import dataclass

import cairosvg
import kagglehub
import torch
from lxml import etree
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


logging.basicConfig(level=logging.INFO)
svg_constraints = kagglehub.package_import('metric/svg-constraints')
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


@dataclass
class Config:
    default_svg: str = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
    timeout_seconds: int = 4 * 60 + 30
    svg_size: int = 384
    font_size: int = 28
    wrap_width: int = 20
    max_svg_size: int = 9990
    prefix: str = "SVG illustration of "
    text_color: str = "gray"
    min_clip_len: int = 70
    max_clip_len: int = 140


@dataclass
class ReturnType:
    score: float = float("-inf")
    color: str = "none"
    clip_len: int = 0
    svg_code: str = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""


class Model:
    def __init__(self):
        self.config = Config()
        self.model_path = kagglehub.model_download("aishikai/google-siglip-so400m-patch14-384/Transformers/default/1")
        self.font_path = kagglehub.dataset_download('tatamikenn/dejavu-fonts-ttf-2-37', path="dejavu-fonts-ttf-2.37/ttf/DejaVuSans-Bold.ttf")
        self.face = freetype.Face(self.font_path)
        self.validator = svg_constraints.SVGConstraints(max_svg_size=self.config.max_svg_size)
        self.evaluator = SVGEvaluator(model_path=self.model_path)
        self.gen_func = partial(
            generate_minimal_svg,
            face=self.face,
            font_size=self.config.font_size,
            svg_size=self.config.svg_size,
            wrap_width=self.config.wrap_width,
            text_color=self.config.text_color,
        )


    def predict(self, description: str, return_type: ReturnType=ReturnType()) -> str:
        def generate_svg():
            try:
                input_text = self.config.prefix + description
                svg_code, score, clip_len, color = self._predict(input_text)
                assert svg_code is not None
                code_size = len(svg_code.encode("utf-8"))

                return_type.score = score
                return_type.color = color
                return_type.clip_len = clip_len
                return_type.svg_code = svg_code
                
                logging.info(f"score={score:.3f}, color={color}, clip_len={clip_len}, code_size={code_size}, input_len={len(input_text)}, input_text: {input_text}")
                return svg_code
            except Exception as e:
                logging.error(f'Exception during SVG generation: {e}')
                return self.config.default_svg

        
        # Execute SVG generation in a new thread to enforce time constraints
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
            future = executor.submit(generate_svg)
            try:
                return future.result(timeout=self.config.timeout_seconds)
            except concurrent.futures.TimeoutError:
                logging.warning("Prediction timed out after %s seconds.", self.config.timeout_seconds)
                return self.config.default_svg
            except Exception as e:
                logging.error(f"An unexpected error occurred: {e}")
                return self.config.default_svg

    
    def _predict(self, text):
        return predict(
            text, 
            evaluator=self.evaluator, 
            validator=self.validator, 
            gen_func=self.gen_func, 
            min_clip_len=self.config.min_clip_len,
            max_clip_len=self.config.max_clip_len,
        )


if __name__ == "__main__":
    display(Config())

## Visualize More Examples

Using the code below, you can see more generated SVGs along with their validation scores.  
**This code works only in interactive mode (not in batch mode).**

In [ ]:
import os


def is_interactive():
    return os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Interactive'


is_interactive()

In [ ]:
def debug():
    texts = [
        "A vibrant orange and pink sunset over a calm ocean, with soft waves gently touching the golden sandy shore.",
        "A vast green meadow under a bright blue sky, dotted with fluffy white clouds and blooming yellow wildflowers.",
        "A bustling cityscape at night, with tall, rectangular skyscrapers glowing in shades of blue and purple lights.",
        "Snow-capped mountains reflecting on a crystal-clear lake, surrounded by dark pine trees under a pastel pink morning sky.",
        "An ancient stone bridge arching over a winding river, with round, smooth rocks and lush green vines hanging from the sides.",
    ]
    total_score = 0
    model = Model()
    for text in texts:
        return_type = ReturnType()
        svg_code = model.predict(text, return_type)        
    
        image = model.evaluator.svg_to_png(svg_code)
        logging.info(f"svg_code: {svg_code}")
        logging.info(f"score: {return_type.score:.4f}")
        display(image)
        total_score += return_type.score
    
    print(f"mean_score: {total_score / len(texts):.4f}")

In [ ]:
%%time
if is_interactive():
    debug()

In [ ]:
%%time
import kaggle_evaluation

kaggle_evaluation.test(Model)